# SCHISM grids and input data

**Learning goals:** Inspect the unstructured mesh and vertical grid, and resolve shared fixture paths reproducibly.

**Prerequisites:** Lesson 2; the shared fixture bundle is downloaded explicitly if absent.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_02_schism_procedural](../journey_02_schism_procedural/)

Next: [journey_04_schism_forcing](../journey_04_schism_forcing/)


## Why this matters: the spatial contract

**Without Rompy:** mesh files, vertical coordinates, boundary numbering, and external-data extents are checked in separate tools, so spatial mismatches can remain hidden until runtime.

**With Rompy:** `SCHISMGrid` keeps the horizontal mesh, vertical grid, and boundary locations together. Later forcing objects use this same spatial contract when extracting data; this lesson focuses only on understanding that contract.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)
print("Mesh:", case / "hgrid.gr3")
print("Vertical grid:", case / "vgrid.in")


## Visual verification: real SCHISM mesh

The mesh is not just a file path: its coordinates and open boundaries control where Rompy samples external forcing.


In [ ]:
import matplotlib.pyplot as plt
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid

grid = SCHISMGrid(hgrid=DataBlob(source=case / "hgrid.gr3"), vgrid=DataBlob(source=case / "vgrid.in"), drag=1)
fig, ax = plt.subplots(figsize=(8, 5))
grid.plot(ax=ax)
ax.set_title("SCHISM regional mesh and open boundary")
plt.show()
print("Nodes:", grid.pylibs_hgrid.np, "elements:", grid.pylibs_hgrid.ne)


## Mesh contract: resolution, boundaries, and vertical coordinates

A SCHISM mesh is an unstructured spatial contract. The node coordinates define where external fields are sampled; element connectivity defines resolution; open-boundary nodes define where ocean and tidal data enter. The vertical grid controls how 3-D HYCOM values map into the water column.


In [ ]:
import numpy as np

hgrid = grid.pylibs_hgrid
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
grid.plot(ax=axes[0])
axes[0].scatter(*grid.boundary_points(), s=4, c="red", label="open boundary")
axes[0].set_title("SCHISM elements and open boundary")
axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude"); axes[0].legend()
axes[1].hist(np.asarray(hgrid.area), bins=20, color="steelblue")
axes[1].set_title("Element-area distribution")
axes[1].set_xlabel("element area (coordinate units²)")
plt.show()
print(f"Node count={hgrid.np}; element count={hgrid.ne}; depth range={hgrid.dp.min():.1f} to {hgrid.dp.max():.1f}")


In [ ]:
vertical_lines = (case / "vgrid.in").read_text().splitlines()
print("Vertical-grid header:", vertical_lines[:4])
print("Vertical levels described:", sum(line.strip() and not line.lstrip().startswith("!") for line in vertical_lines))
print("Interpretation: inspect the vgrid.in convention before mapping HYCOM depth levels; this check does not validate vertical interpolation.")
